# Grammar Scoring Engine for Spoken Responses

This notebook predicts the continuous grammar-MOS label (0–5) from each WAV file. It combines an English ASR transcript with acoustic fluency features. The validation score reported below is out-of-fold (OOF), so it is a more honest benchmark than fitting and scoring on the same 769 rows.

**Metrics:** RMSE is minimized and Pearson correlation is maximized, matching the leaderboard. The final cell writes `submission.csv`.

## Setup

Run this once in Kaggle with Internet enabled, or attach a Kaggle Dataset containing the Faster-Whisper model and point `MODEL_SIZE_OR_PATH` at it. The install line is deliberately separate so an environment that already has the packages does not need network access.

In [ ]:
# Uncomment only if these packages are not already available.
# !pip install -q faster-whisper librosa soundfile scikit-learn pandas seaborn transformers

from pathlib import Path
import re
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import librosa
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
from scipy.stats import pearsonr
from scipy.sparse import hstack, csr_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.ensemble import ExtraTreesRegressor, RandomForestRegressor
from sklearn.linear_model import Ridge, RidgeCV
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler

SEED = 42
N_SPLITS = 5
MODEL_SIZE_OR_PATH = 'distil-large-v3'  # stronger English ASR; use a local attached Faster-Whisper model path when Internet is off
# Locate the data both locally and in a Kaggle competition notebook.
local_root = Path('Dataset_Final')
kaggle_roots = [p.parent for p in Path('/kaggle/input').glob('**/train.csv')] if Path('/kaggle/input').exists() else []
DATA_ROOT = local_root if (local_root / 'train.csv').exists() else (kaggle_roots[0] if kaggle_roots else None)
assert DATA_ROOT is not None, 'Could not find train.csv. Attach the competition data under Add Input.'
print(f'Using data at: {DATA_ROOT}')
CACHE_DIR = Path('cache'); CACHE_DIR.mkdir(exist_ok=True)


In [ ]:
train = pd.read_csv(DATA_ROOT / 'train.csv')
test = pd.read_csv(DATA_ROOT / 'test.csv')
sample = pd.read_csv(DATA_ROOT / 'sample_submission.csv')
assert {'filename', 'label'} <= set(train.columns)
assert {'filename', 'label'} <= set(test.columns)
assert train['filename'].is_unique and test['filename'].is_unique
print(f'train: {train.shape}; test: {test.shape}')
if len(sample) != len(test) or set(sample.filename) != set(test.filename):
    print('Warning: sample_submission.csv does not match test.csv. Submission will correctly use test.csv order.')

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(train.label, bins=np.arange(-.25, 5.76, .5), ax=ax[0], color='#376996')
ax[0].set(title='Training label distribution', xlabel='Grammar MOS')
sns.boxplot(data=train, y='label', ax=ax[1], color='#91c7b1')
ax[1].set(title='Target range check', ylabel='Grammar MOS')
plt.tight_layout()
train.head()


## Audio preprocessing and features

All audio is converted to mono 16 kHz. We retain duration, energy, silence ratio, zero-crossing rate, spectral centroid, and MFCC summary statistics. These do not claim to measure grammar directly; they capture delivery properties (pauses, consistency, and recording quality) that complement transcript-based evidence.

In [ ]:
def audio_features(path):
    y, sr = librosa.load(path, sr=16_000, mono=True)
    if len(y) == 0:
        return {'duration_s': 0.0}
    rms = librosa.feature.rms(y=y, frame_length=1024, hop_length=512)[0]
    zcr = librosa.feature.zero_crossing_rate(y, frame_length=1024, hop_length=512)[0]
    centroid = librosa.feature.spectral_centroid(y=y, sr=sr, n_fft=1024, hop_length=512)[0]
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13, n_fft=1024, hop_length=512)
    out = {
        'duration_s': len(y) / sr, 'rms_mean': rms.mean(), 'rms_std': rms.std(),
        'silence_ratio': (rms < max(0.005, np.quantile(rms, .15))).mean(),
        'zcr_mean': zcr.mean(), 'zcr_std': zcr.std(),
        'centroid_mean': centroid.mean(), 'centroid_std': centroid.std(),
    }
    out.update({f'mfcc_{i}_mean': mfcc[i].mean() for i in range(13)})
    out.update({f'mfcc_{i}_std': mfcc[i].std() for i in range(13)})
    return out

def make_audio_frame(frame, split):
    rows = []
    for filename in tqdm(frame.filename, desc=f'{split} acoustic features'):
        rows.append(audio_features(DATA_ROOT / split / filename))
    return pd.DataFrame(rows)

train_audio = make_audio_frame(train, 'train')
test_audio = make_audio_frame(test, 'test')
numeric_cols = list(train_audio.columns)
fig, ax = plt.subplots(figsize=(7, 4))
sns.scatterplot(x=train_audio.duration_s, y=train.label, alpha=.7, ax=ax)
ax.set(title='Duration versus grammar label', xlabel='Duration (seconds)', ylabel='Grammar MOS')
plt.show()


## Speech-to-text (cached)

Grammar is primarily a linguistic property, so transcripts are the central feature. Faster-Whisper is run once and saved in `cache/transcripts.csv`; reruns reuse that file. `beam_size=5` improves transcription consistency without using labels.

In [ ]:
from faster_whisper import WhisperModel
import torch

# Cache is keyed by model name, preventing accidental reuse of weaker-model transcripts.
model_cache_name = re.sub(r'[^A-Za-z0-9_.-]+', '_', Path(MODEL_SIZE_OR_PATH).name)
cache_file = CACHE_DIR / f'transcripts_{model_cache_name}.csv'
all_files = pd.concat([
    train.assign(split='train')[['filename', 'split']],
    test.assign(split='test')[['filename', 'split']]
], ignore_index=True)

if cache_file.exists():
    transcripts = pd.read_csv(cache_file)
else:
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    compute_type = 'float16' if device == 'cuda' else 'int8'
    asr = WhisperModel(MODEL_SIZE_OR_PATH, device=device, compute_type=compute_type)
    records = []
    for row in tqdm(all_files.itertuples(index=False), total=len(all_files), desc='ASR'):
        segments, info = asr.transcribe(str(DATA_ROOT / row.split / row.filename), language='en', beam_size=5, vad_filter=True)
        text = ' '.join(s.text.strip() for s in segments).strip()
        records.append({'filename': row.filename, 'split': row.split, 'transcript': text,
                        'asr_language_probability': info.language_probability})
    transcripts = pd.DataFrame(records)
    transcripts.to_csv(cache_file, index=False)

assert len(transcripts) == len(all_files) and transcripts.transcript.notna().all()
train_text = transcripts.query("split == 'train'").set_index('filename').loc[train.filename].reset_index()
test_text = transcripts.query("split == 'test'").set_index('filename').loc[test.filename].reset_index()
display(train_text[['filename', 'transcript']].head(3))


## Deep speech and grammatical-acceptability features

WavLM adds pretrained speech representations that capture delivery, fluency, pronunciation, and other information absent from summary MFCC features. A RoBERTa model fine-tuned on CoLA scores the grammatical acceptability of transcript sentences. Both feature sets are cached.

In [ ]:
# If this import fails, run: !pip install -q transformers
from transformers import AutoFeatureExtractor, AutoModel, AutoTokenizer, AutoModelForSequenceClassification
import gc

deep_cache = CACHE_DIR / f'wavlm_cola_features_{model_cache_name}_v1.npz'
if deep_cache.exists():
    saved = np.load(deep_cache)
    train_wavlm, test_wavlm = saved['train_wavlm'], saved['test_wavlm']
    train_cola, test_cola = saved['train_cola'], saved['test_cola']
else:
    # Release the ASR model before loading two PyTorch models on the GPU.
    if 'asr' in globals():
        del asr
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    device = 'cuda' if torch.cuda.is_available() else 'cpu'

    wav_name = 'microsoft/wavlm-base-plus'
    wav_processor = AutoFeatureExtractor.from_pretrained(wav_name)
    wav_model = AutoModel.from_pretrained(wav_name).to(device).eval()

    @torch.inference_mode()
    def wavlm_embedding(path, seconds=10, batch_size=2):
        y_audio, _ = librosa.load(path, sr=16_000, mono=True)
        width = seconds * 16_000
        chunks = [y_audio[i:i+width] for i in range(0, len(y_audio), width)] or [np.zeros(width, dtype=np.float32)]
        chunks = [np.pad(x, (0, width-len(x))) if len(x) < width else x for x in chunks]
        vectors = []
        for start in range(0, len(chunks), batch_size):
            batch = wav_processor(chunks[start:start+batch_size], sampling_rate=16_000, return_tensors='pt', padding=True)
            values = batch.input_values.to(device)
            hidden = wav_model(values).last_hidden_state
            vectors.append(hidden.mean(dim=1).float().cpu().numpy())
        z = np.vstack(vectors)
        return np.concatenate([z.mean(axis=0), z.std(axis=0)])

    def embed_split(frame, split):
        return np.vstack([wavlm_embedding(DATA_ROOT / split / f) for f in tqdm(frame.filename, desc=f'{split} WavLM')])

    train_wavlm = embed_split(train, 'train')
    test_wavlm = embed_split(test, 'test')
    del wav_model
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

    cola_name = 'textattack/roberta-base-CoLA'
    cola_tokenizer = AutoTokenizer.from_pretrained(cola_name)
    cola_model = AutoModelForSequenceClassification.from_pretrained(cola_name).to(device).eval()

    @torch.inference_mode()
    def cola_features(texts, batch_size=32):
        sentences, owners = [], []
        for owner, text_value in enumerate(texts):
            parts = [s.strip() for s in re.split(r'(?<=[.!?])\s+', str(text_value)) if len(s.split()) >= 3]
            if not parts: parts = [str(text_value).strip() or 'No response.']
            for part in parts:
                words = part.split()
                for start in range(0, len(words), 80):
                    sentences.append(' '.join(words[start:start+80])); owners.append(owner)
        scores = []
        for start in tqdm(range(0, len(sentences), batch_size), desc='CoLA grammar scores'):
            enc = cola_tokenizer(sentences[start:start+batch_size], padding=True, truncation=True, max_length=128, return_tensors='pt').to(device)
            prob = torch.softmax(cola_model(**enc).logits, dim=-1)[:, 1]
            scores.extend(prob.float().cpu().numpy())
        scores = np.asarray(scores)
        rows = []
        for owner in range(len(texts)):
            v = scores[np.asarray(owners) == owner]
            rows.append([v.mean(), v.std(), v.min(), v.max(), (v >= .5).mean(), len(v)])
        return np.asarray(rows, dtype=float)

    train_cola = cola_features(train_text.transcript.fillna('').to_numpy())
    test_cola = cola_features(test_text.transcript.fillna('').to_numpy())
    np.savez_compressed(deep_cache, train_wavlm=train_wavlm, test_wavlm=test_wavlm, train_cola=train_cola, test_cola=test_cola)
    del cola_model
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

print('Deep feature shapes:', train_wavlm.shape, train_cola.shape, test_wavlm.shape, test_cola.shape)


## Modeling and cross-validation

A word TF–IDF representation captures vocabulary and common grammatical patterns; character n-grams are robust to ASR misspellings and partial words. Acoustic features are standardized and appended. Ridge regression is intentionally regularized because 769 labeled recordings is a small sample. The folds are shuffled and fixed with a seed for reproducibility.

In [ ]:
def transcript_stats(texts):
    # Lightweight, interpretable indicators of response complexity and disfluency.
    rows = []
    fillers = {'um', 'uh', 'erm', 'ah', 'like'}
    for text in texts:
        words = re.findall(r"[a-zA-Z']+", str(text).lower())
        n = len(words)
        sentences = max(1, len(re.findall(r'[.!?]+', str(text))))
        repeats = sum(words[i] == words[i-1] for i in range(1, n))
        rows.append([
            n, len(set(words)), len(set(words)) / max(n, 1),
            np.mean([len(w) for w in words]) if n else 0,
            n / sentences, sentences, sum(w in fillers for w in words),
            repeats, text.count(','), text.count('?'), text.count('!')
        ])
    return np.asarray(rows, dtype=float)

def linguistic_features(text_train, text_valid):
    word = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_df=.98, sublinear_tf=True, max_features=30_000)
    char = TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 5), min_df=2, sublinear_tf=True, max_features=40_000)
    return hstack([word.fit_transform(text_train), char.fit_transform(text_train)]), hstack([word.transform(text_valid), char.transform(text_valid)]), word, char

y = train.label.to_numpy(float)
texts = train_text.transcript.fillna('').to_numpy()
X_num = train_audio[numeric_cols].to_numpy()
X_stats = transcript_stats(texts)
X_tree = np.column_stack([X_num, X_stats])
X_deep = np.column_stack([train_wavlm, train_cola])
cv = KFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
ridge_oof = np.zeros(len(train))
tree_oof = np.zeros(len(train))
forest_oof = np.zeros(len(train))
deep_oof = np.zeros(len(train))
for fold, (tr, va) in enumerate(cv.split(train), 1):
    Xtr_text, Xva_text, _, _ = linguistic_features(texts[tr], texts[va])
    imp = SimpleImputer(strategy='median').fit(X_num[tr])
    scale = StandardScaler().fit(imp.transform(X_num[tr]))
    Xtr = hstack([Xtr_text, csr_matrix(scale.transform(imp.transform(X_num[tr])))])
    Xva = hstack([Xva_text, csr_matrix(scale.transform(imp.transform(X_num[va])))])
    ridge = Ridge(alpha=18.0).fit(Xtr, y[tr])
    ridge_oof[va] = ridge.predict(Xva)
    tree_imp = SimpleImputer(strategy='median').fit(X_tree[tr])
    tree = ExtraTreesRegressor(n_estimators=600, min_samples_leaf=3, max_features=.8, random_state=SEED + fold, n_jobs=-1)
    tree.fit(tree_imp.transform(X_tree[tr]), y[tr])
    tree_oof[va] = tree.predict(tree_imp.transform(X_tree[va]))
    forest = RandomForestRegressor(n_estimators=700, min_samples_leaf=3, max_features=.7, random_state=100 + fold, n_jobs=-1)
    forest.fit(tree_imp.transform(X_tree[tr]), y[tr])
    forest_oof[va] = forest.predict(tree_imp.transform(X_tree[va]))
    deep_imp = SimpleImputer(strategy='median').fit(X_deep[tr])
    deep_scale = StandardScaler().fit(deep_imp.transform(X_deep[tr]))
    deep_model = RidgeCV(alphas=np.logspace(1, 4, 10)).fit(deep_scale.transform(deep_imp.transform(X_deep[tr])), y[tr])
    deep_oof[va] = deep_model.predict(deep_scale.transform(deep_imp.transform(X_deep[va])))
    print(f'Fold {fold}: Ridge={mean_squared_error(y[va], ridge_oof[va]) ** .5:.4f} | ExtraTrees={mean_squared_error(y[va], tree_oof[va]) ** .5:.4f} | RF={mean_squared_error(y[va], forest_oof[va]) ** .5:.4f} | WavLM+CoLA={mean_squared_error(y[va], deep_oof[va]) ** .5:.4f}')

# Select the blend only from out-of-fold predictions; do not use leaderboard feedback.
blend_candidates = []
for ridge_w in np.linspace(0, .4, 9):
    for tree_w in np.linspace(0, 1 - ridge_w, 21):
        forest_w = 1 - ridge_w - tree_w
        blend_candidates.append((mean_squared_error(y, ridge_w * ridge_oof + tree_w * tree_oof + forest_w * forest_oof), ridge_w, tree_w, forest_w))
_, ridge_weight, tree_weight, forest_weight = min(blend_candidates)
base_oof = ridge_weight * ridge_oof + tree_weight * tree_oof + forest_weight * forest_oof
deep_weight = min(np.linspace(0, 1.0, 41), key=lambda w: mean_squared_error(y, (1-w) * base_oof + w * deep_oof))
oof = (1-deep_weight) * base_oof + deep_weight * deep_oof
oof_rmse = mean_squared_error(y, oof) ** .5
oof_pearson = pearsonr(y, oof).statistic
print(f'\nSelected blend weights — Ridge: {ridge_weight:.2f}, ExtraTrees: {tree_weight:.2f}, RandomForest: {forest_weight:.2f}')
print(f'Selected deep-feature blend weight: {deep_weight:.3f}')
print(f'Required training-data benchmark (5-fold OOF RMSE): {oof_rmse:.4f}')
print(f'5-fold OOF Pearson correlation: {oof_pearson:.4f}')

fig, ax = plt.subplots(figsize=(5, 5))
sns.scatterplot(x=y, y=oof, alpha=.75, ax=ax)
ax.plot([0, 5], [0, 5], '--', color='black')
ax.set(xlim=(0,5), ylim=(0,5), xlabel='True MOS', ylabel='OOF prediction', title=f'OOF predictions | RMSE={oof_rmse:.3f}, r={oof_pearson:.3f}')
plt.show()


## Train on all labeled audio and create submission

After evaluation, vectorizers and model are refit using every training item. Predictions are clipped to the rubric interval `[0, 5]`. The submission uses `test.csv` filenames and row order, which is the authoritative evaluation list.

In [ ]:
test_texts = test_text.transcript.fillna('').to_numpy()
Xtr_text, Xte_text, _, _ = linguistic_features(texts, test_texts)
imp = SimpleImputer(strategy='median').fit(X_num)
scale = StandardScaler().fit(imp.transform(X_num))
Xtr = hstack([Xtr_text, csr_matrix(scale.transform(imp.transform(X_num)))])
Xte_num = test_audio[numeric_cols].to_numpy()
Xte = hstack([Xte_text, csr_matrix(scale.transform(imp.transform(Xte_num)))])
final_model = Ridge(alpha=18.0).fit(Xtr, y)
ridge_test_pred = final_model.predict(Xte)
X_tree_test = np.column_stack([Xte_num, transcript_stats(test_texts)])
tree_imp = SimpleImputer(strategy='median').fit(X_tree)
tree_model = ExtraTreesRegressor(n_estimators=800, min_samples_leaf=3, max_features=.8, random_state=SEED, n_jobs=-1)
tree_model.fit(tree_imp.transform(X_tree), y)
tree_test_pred = tree_model.predict(tree_imp.transform(X_tree_test))
forest_model = RandomForestRegressor(n_estimators=900, min_samples_leaf=3, max_features=.7, random_state=100, n_jobs=-1)
forest_model.fit(tree_imp.transform(X_tree), y)
forest_test_pred = forest_model.predict(tree_imp.transform(X_tree_test))
base_test_pred = ridge_weight * ridge_test_pred + tree_weight * tree_test_pred + forest_weight * forest_test_pred
X_deep_test = np.column_stack([test_wavlm, test_cola])
deep_imp = SimpleImputer(strategy='median').fit(X_deep)
deep_scale = StandardScaler().fit(deep_imp.transform(X_deep))
deep_model = RidgeCV(alphas=np.logspace(1, 4, 10)).fit(deep_scale.transform(deep_imp.transform(X_deep)), y)
deep_test_pred = deep_model.predict(deep_scale.transform(deep_imp.transform(X_deep_test)))
test_pred = np.clip((1-deep_weight) * base_test_pred + deep_weight * deep_test_pred, 0, 5)
submission = pd.DataFrame({'filename': test.filename, 'label': test_pred})
assert len(submission) == len(test)
assert submission.filename.equals(test.filename)
assert submission.label.between(0, 5).all()
submission.to_csv('submission.csv', index=False)
display(submission.head())
print('Wrote submission.csv:', submission.shape)


## Brief report

**Pipeline.** WAV recordings are resampled to 16 kHz mono. Acoustic summaries describe duration, energy, quiet frames, spectral shape, and MFCCs. Faster-Whisper produces an English transcript, which is cached to make experimentation repeatable and affordable. Word and character TF–IDF features plus standardized acoustics feed a regularized Ridge regressor.

**Validation.** Five shuffled K-fold predictions are generated without training on their held-out row. The notebook prints the required training benchmark as 5-fold OOF RMSE and Pearson correlation; these values should be copied into the final competition notes after execution.

**Interpretation and limitations.** Text n-grams provide inspectable linguistic evidence, while acoustic features are supplementary rather than a proxy for grammar. Scores may be affected by ASR errors, accents, or recording conditions. A strong next experiment is to replace `small.en` with an attached `distil-large-v3` Faster-Whisper model, retain exactly the same OOF folds, and compare both metrics before selecting a final submission.